# Analyse des Résultats d'Alignement

Ce notebook permet d'analyser les résultats du pipeline d'alignement des taxonomies animales.

In [ ]:
import pandas as pd
import numpy as np
from owlready2 import *
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Chargement des Données

In [ ]:
# Chargement des clusters d'alignement
clusters_df = pd.read_csv('Pipeline/animal_clusters.csv')
print(f"📊 Nombre d'alignements trouvés : {len(clusters_df)}")
clusters_df.head()

In [ ]:
# Chargement des métadonnées complètes
metadata_df = pd.read_csv('Pipeline/metadata.csv')
print(f"📚 Nombre total de concepts : {len(metadata_df)}")
print(f"   - TAXONOMY_A : {len(metadata_df[metadata_df['source'] == 'TAXONOMY_A'])}")
print(f"   - TAXONOMY_B : {len(metadata_df[metadata_df['source'] == 'TAXONOMY_B'])}")

## 2. Analyse des Alignements

In [ ]:
# Distribution des tailles de clusters
plt.figure(figsize=(10, 5))
clusters_df['size'].value_counts().sort_index().plot(kind='bar')
plt.title('Distribution des Tailles de Clusters')
plt.xlabel('Nombre de concepts dans le cluster')
plt.ylabel('Fréquence')
plt.show()

print(f"Taille moyenne des clusters : {clusters_df['size'].mean():.2f}")
print(f"Taille médiane : {clusters_df['size'].median():.0f}")

In [ ]:
# Exemples d'alignements
print("🎯 Exemples d'alignements détectés :\n")
for idx, row in clusters_df.head(10).iterrows():
    print(f"Cluster {row['cluster_id']}:")
    print(f"  • TAXONOMY_A : {row['taxonomy_a_labels']}")
    print(f"  • TAXONOMY_B : {row['taxonomy_b_labels']}")
    print()

## 3. Analyse de la Hiérarchie

In [ ]:
# Concepts avec multi-héritage
def count_parents(parent_str):
    if pd.isna(parent_str) or parent_str == '':
        return 0
    return len([p for p in str(parent_str).split('|') if p.strip()])

metadata_df['num_parents'] = metadata_df['parent_labels'].apply(count_parents)

plt.figure(figsize=(10, 5))
metadata_df['num_parents'].value_counts().sort_index().plot(kind='bar')
plt.title('Distribution du Nombre de Parents par Concept')
plt.xlabel('Nombre de parents')
plt.ylabel('Nombre de concepts')
plt.show()

multi_heritage = metadata_df[metadata_df['num_parents'] > 1]
print(f"📊 Concepts avec multi-héritage : {len(multi_heritage)} ({len(multi_heritage)/len(metadata_df)*100:.1f}%)")

## 4. Analyse de la Méta-Ontologie

In [ ]:
# Chargement de la méta-ontologie
meta_onto = get_ontology('Pipeline/meta_animal_taxonomy.owl').load()

print(f"🏗️ Méta-Ontologie :")
print(f"   - Classes créées : {len(list(meta_onto.classes()))}")
print(f"   - Propriétés : {len(list(meta_onto.properties()))}")
print(f"   - Individus : {len(list(meta_onto.individuals()))}")

In [ ]:
# Exploration de quelques classes
print("\n🔍 Exemples de Meta-Classes :\n")
for i, cls in enumerate(list(meta_onto.classes())[:10]):
    if hasattr(cls, 'label') and cls.label:
        label = cls.label[0] if cls.label else cls.name
        parents = [p.label[0] if hasattr(p, 'label') and p.label else p.name 
                   for p in cls.is_a if hasattr(p, 'name')]
        print(f"{i+1}. {label}")
        print(f"   Parents : {', '.join(parents[:3])}")
        if hasattr(cls, 'hasSourceURI') and cls.hasSourceURI:
            print(f"   URIs sources : {len(cls.hasSourceURI)}")
        print()

## 5. Statistiques des Embeddings

In [ ]:
# Chargement des embeddings
embeddings = np.load('Pipeline/embeddings.npy')
print(f"📐 Forme des embeddings : {embeddings.shape}")
print(f"   - Nombre de concepts : {embeddings.shape[0]}")
print(f"   - Dimensions : {embeddings.shape[1]}")
print(f"   - Norme moyenne : {np.linalg.norm(embeddings, axis=1).mean():.4f}")

In [ ]:
# Distribution des normes
norms = np.linalg.norm(embeddings, axis=1)
plt.figure(figsize=(10, 5))
plt.hist(norms, bins=50, edgecolor='black')
plt.title('Distribution des Normes des Vecteurs d\'Embeddings')
plt.xlabel('Norme L2')
plt.ylabel('Fréquence')
plt.axvline(norms.mean(), color='red', linestyle='--', label=f'Moyenne: {norms.mean():.2f}')
plt.legend()
plt.show()

## 6. Analyse des Concepts Non Alignés

In [ ]:
# URIs alignés
aligned_uris = set()
for idx, row in clusters_df.iterrows():
    aligned_uris.update([u.strip() for u in str(row['taxonomy_a_uris']).split(' | ') if u.strip()])
    aligned_uris.update([u.strip() for u in str(row['taxonomy_b_uris']).split(' | ') if u.strip()])

# Concepts non alignés
non_aligned = metadata_df[~metadata_df['uri'].isin(aligned_uris)]

print(f"📊 Statistiques d'alignement :")
print(f"   - Concepts alignés : {len(aligned_uris)}")
print(f"   - Concepts non alignés : {len(non_aligned)}")
print(f"   - Taux d'alignement : {len(aligned_uris)/len(metadata_df)*100:.1f}%")
print(f"\n   Par source :")
print(f"   - TAXONOMY_A non alignés : {len(non_aligned[non_aligned['source'] == 'TAXONOMY_A'])}")
print(f"   - TAXONOMY_B non alignés : {len(non_aligned[non_aligned['source'] == 'TAXONOMY_B'])}")

In [ ]:
# Exemples de concepts non alignés
print("\n🔍 Exemples de concepts non alignés :\n")
for idx, row in non_aligned.head(10).iterrows():
    print(f"• [{row['source']}] {row['label']}")
    if row['definition']:
        print(f"  → {row['definition'][:100]}...")
    print()

## 7. Export des Résultats

In [ ]:
# Créer un rapport texte
report = f"""
RAPPORT D'ALIGNEMENT DES TAXONOMIES ANIMALES
{'='*60}

1. Statistiques Générales
   - Concepts totaux : {len(metadata_df)}
   - TAXONOMY_A : {len(metadata_df[metadata_df['source'] == 'TAXONOMY_A'])}
   - TAXONOMY_B : {len(metadata_df[metadata_df['source'] == 'TAXONOMY_B'])}
   
2. Alignements
   - Clusters d'alignement : {len(clusters_df)}
   - Concepts alignés : {len(aligned_uris)}
   - Taux d'alignement : {len(aligned_uris)/len(metadata_df)*100:.1f}%
   - Taille moyenne cluster : {clusters_df['size'].mean():.2f}
   
3. Méta-Ontologie
   - Classes créées : {len(list(meta_onto.classes()))}
   - Multi-héritage : {len(multi_heritage)} concepts ({len(multi_heritage)/len(metadata_df)*100:.1f}%)
   
4. Embeddings
   - Dimensions : {embeddings.shape[1]}
   - Norme moyenne : {norms.mean():.4f}
"""

print(report)

# Sauvegarder le rapport
with open('Pipeline/rapport_analyse.txt', 'w', encoding='utf-8') as f:
    f.write(report)
    
print("\n✅ Rapport sauvegardé dans Pipeline/rapport_analyse.txt")